# Surveiller la dérive des données avec Evidently

Ce notebook met en place un **workflow de surveillance** du modèle d'attrition, tel qu'il tournerait à chaque nouveau lot de données de production :

1. **Contrôler la qualité** du lot (valeurs manquantes, modalités inconnues) : un bug de pipeline ressemble à une dérive, mais ne se traite pas de la même façon ;
2. **Mesurer la dérive** de chaque variable et des prédictions par rapport à une référence figée (PSI) ;
3. **Décider** : statut OK / avertissement / critique, et réentraînement seulement si la dérive critique persiste ;
4. **Tracer** le tout dans MLflow (métriques, statut, rapport HTML Evidently).

La production est **simulée** : on tire des lots hebdomadaires dans une partie du jeu de données mise de côté, en y injectant progressivement une dérive, puis un bug de données.

**Prérequis**

```bash
uv add evidently
uv run mlflow server --host 127.0.0.1 --port 5000   # dans un terminal dédié
```

## 1. Imports et configuration

In [ ]:
import os
import tempfile
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from evidently import DataDefinition, Dataset, Report
from evidently.metrics import DatasetMissingValueCount, OutListValueCount
from evidently.presets import DataDriftPreset
from evidently.tests import eq
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

from training.data import load_data

In [ ]:
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT = "employee_attrition_monitoring"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)

TARGET = "LeaveOrNot"
PREDICTION = "proba_depart"
NUMERICAL = ["JoiningYear", "Age", "ExperienceInCurrentDomain"]
CATEGORICAL = ["Education", "PaymentTier", "Gender", "EverBenched", "City"]
FEATURES = NUMERICAL + CATEGORICAL

# Règles de décision (repères usuels pour le PSI)
PSI_WARNING = 0.1  # à examiner
PSI_CRITICAL = 0.2  # dérive significative
CRITICAL_BATCHES_BEFORE_RETRAIN = 2  # la dérive critique doit persister sur 2 lots consécutifs

## 2. Modèle et jeu de référence

Le jeu de données est coupé en trois :

- **entraînement** : sert à ajuster le modèle ;
- **référence** : données non vues par le modèle, figées, auxquelles on comparera la production. On ne prend pas les données d'entraînement, sur lesquelles les prédictions sont trop optimistes ;
- **production** : réservoir dans lequel on tirera les lots simulés.

In [ ]:
df = load_data("../data/employees.csv")

train_df, rest_df = train_test_split(df, test_size=0.6, stratify=df[TARGET], random_state=42)
reference_df, production_pool = train_test_split(
    rest_df, test_size=2 / 3, stratify=rest_df[TARGET], random_state=42
)

model = Pipeline(
    [
        (
            "preprocessor",
            ColumnTransformer(
                [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL)],
                remainder="passthrough",
            ),
        ),
        ("classifier", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)),
    ]
).fit(train_df[FEATURES], train_df[TARGET])


def score(data: pd.DataFrame) -> pd.DataFrame:
    # Ajoute la probabilité de départ prédite par le modèle.
    scored = data.copy()
    scored[PREDICTION] = model.predict_proba(data[FEATURES])[:, 1]
    return scored


reference = score(reference_df)
print(f"entraînement : {len(train_df)} lignes, référence : {len(reference)}, production : {len(production_pool)}")
print(f"AUC sur la référence : {roc_auc_score(reference[TARGET], reference[PREDICTION]):.3f}")

## 3. Simulation des lots de production

Six semaines de 500 lignes chacune :

- **S1, S2** : même population que la référence ;
- **S3 à S5** : vague de recrutements de profils jeunes à Pune, de plus en plus marquée (la composition de la population change : *data drift*) ;
- **S6** : population normale, mais un système amont se met à envoyer le genre en minuscules pour une partie des lignes (*bug de données*).

In [ ]:
def make_batch(intensity: float, seed: int, n: int = 500) -> pd.DataFrame:
    # Tire un lot dans le réservoir en sur-représentant les jeunes et la ville de Pune.
    weights = np.where(production_pool["Age"] <= 26, 1 + 4 * intensity, 1.0) * np.where(
        production_pool["City"] == "Pune", 1 + 2 * intensity, 1.0
    )
    return production_pool.sample(n, replace=True, weights=weights, random_state=seed).reset_index(drop=True)


batches = {
    "S1": make_batch(0.0, seed=1),
    "S2": make_batch(0.0, seed=2),
    "S3": make_batch(0.15, seed=3),
    "S4": make_batch(1.0, seed=4),
    "S5": make_batch(1.0, seed=5),
    "S6": make_batch(0.0, seed=6),
}

# Bug amont en S6 : le genre arrive en minuscules pour 30 % des lignes
bugged = batches["S6"].sample(frac=0.3, random_state=0).index
batches["S6"].loc[bugged, "Gender"] = batches["S6"].loc[bugged, "Gender"].str.lower()

## 4. Les briques du workflow

Evidently travaille sur des `Dataset` : un DataFrame accompagné d'une `DataDefinition` qui déclare le type de chaque colonne. Un `Report` est une liste de métriques ; `report.run(courant, référence)` renvoie un résultat que l'on peut afficher, exporter en HTML ou lire comme un dictionnaire.

In [ ]:
DATA_DEFINITION = DataDefinition(
    numerical_columns=NUMERICAL + [PREDICTION],
    categorical_columns=CATEGORICAL,
)
MONITORED = FEATURES + [PREDICTION]  # la cible n'est pas surveillée : en production elle arrive plus tard


def to_dataset(data: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(data[MONITORED], data_definition=DATA_DEFINITION)


REFERENCE_DATASET = to_dataset(reference)
ALLOWED_VALUES = {c: sorted(reference[c].unique().tolist()) for c in CATEGORICAL}

In [ ]:
def check_quality(batch: pd.DataFrame) -> list[str]:
    # Étape 1 : tests de qualité. Renvoie la liste des tests en échec (vide si tout va bien).
    report = Report(
        [DatasetMissingValueCount(tests=[eq(0)])]
        + [OutListValueCount(column=c, values=ALLOWED_VALUES[c], tests=[eq(0)]) for c in CATEGORICAL]
    )
    snapshot = report.run(to_dataset(batch), None)
    return [
        test["description"]
        for test in snapshot.dict()["tests"]
        if getattr(test["status"], "value", test["status"]) == "FAIL"
    ]


def check_drift(batch: pd.DataFrame):
    # Étape 2 : PSI de chaque colonne surveillée par rapport à la référence.
    report = Report([DataDriftPreset(method="psi", threshold=PSI_WARNING)])
    snapshot = report.run(to_dataset(batch), REFERENCE_DATASET)
    psi = {
        metric["config"]["column"]: float(metric["value"])
        for metric in snapshot.dict()["metrics"]
        if metric["config"]["type"].endswith("ValueDrift")
    }
    return snapshot, psi


def drift_status(psi: dict[str, float]) -> str:
    # Étape 3 : statut du lot d'après la colonne qui dérive le plus.
    worst = max(psi.values())
    if worst >= PSI_CRITICAL:
        return "critique"
    if worst >= PSI_WARNING:
        return "avertissement"
    return "ok"

In [ ]:
def monitor_batch(name: str, batch: pd.DataFrame) -> dict:
    # Exécute le workflow sur un lot et trace le résultat dans un run MLflow.
    scored = score(batch)
    quality_failures = check_quality(scored)
    snapshot, psi = check_drift(scored)
    # Un lot en échec qualité n'est pas interprété comme une dérive : c'est le pipeline qu'il faut corriger
    status = "bug de données" if quality_failures else drift_status(psi)

    with mlflow.start_run(run_name=name):
        mlflow.set_tags({"status": status, "drift_method": "psi"})
        mlflow.log_params({"n_rows": len(batch), "psi_warning": PSI_WARNING, "psi_critical": PSI_CRITICAL})
        mlflow.log_metrics({f"psi_{column}": value for column, value in psi.items()})
        mlflow.log_metrics(
            {
                "psi_max": max(psi.values()),
                "drifted_columns_share": float(np.mean([v >= PSI_WARNING for v in psi.values()])),
                "quality_failures": len(quality_failures),
                # Disponible seulement quand les vraies étiquettes sont connues, donc en différé
                "auc": roc_auc_score(scored[TARGET], scored[PREDICTION]),
            }
        )
        if quality_failures:
            mlflow.log_text("\n".join(quality_failures), "quality_failures.txt")
        with tempfile.TemporaryDirectory() as tmp:
            html_path = Path(tmp) / f"drift_report_{name}.html"
            snapshot.save_html(str(html_path))
            mlflow.log_artifact(str(html_path), artifact_path="evidently")

    return {"lot": name, "statut": status, "qualité": quality_failures, "psi": psi, "snapshot": snapshot,
            "auc": roc_auc_score(scored[TARGET], scored[PREDICTION])}

## 5. Exécution lot par lot

En production, cette boucle serait un script planifié (cron, tâche CI, orchestrateur) lancé à l'arrivée de chaque lot. La décision de réentraîner n'est prise que si la dérive critique **persiste**.

In [ ]:
results = {}
consecutive_critical = 0

for name, batch in batches.items():
    result = monitor_batch(name, batch)
    consecutive_critical = consecutive_critical + 1 if result["statut"] == "critique" else 0

    if result["statut"] == "bug de données":
        action = "corriger le pipeline amont"
    elif consecutive_critical >= CRITICAL_BATCHES_BEFORE_RETRAIN:
        action = "déclencher le réentraînement"
    elif result["statut"] != "ok":
        action = "examiner les variables en dérive"
    else:
        action = "aucune"

    result["action"] = action
    results[name] = result
    print(f"{name} : {result['statut']:<15} -> {action}")

## 6. Lecture des résultats

Le tableau donne le PSI de chaque colonne par lot. À lire avec les seuils : 0,1 (à examiner) et 0,2 (dérive significative).

Comparer aussi les colonnes de PSI à la colonne `auc` : dans cette simulation, l'AUC ne baisse pas pendant la dérive (S3 à S5). La population change, mais la relation entre les variables et le départ reste la même : **une dérive des données n'est pas une dégradation du modèle**, c'est un signal à examiner.

In [ ]:
psi_table = pd.DataFrame({name: r["psi"] for name, r in results.items()}).T
summary = psi_table.round(3).assign(
    auc=[round(r["auc"], 3) for r in results.values()],
    statut=[r["statut"] for r in results.values()],
    action=[r["action"] for r in results.values()],
)
summary

In [ ]:
# Détail des tests de qualité en échec
for name, r in results.items():
    for failure in r["qualité"]:
        print(f"{name} : {failure}")

Le rapport Evidently d'un lot s'affiche directement dans le notebook : distributions de référence et courantes côte à côte pour chaque colonne. Le même rapport est enregistré en HTML dans les artefacts du run MLflow correspondant.

In [ ]:
results["S4"]["snapshot"]

## 7. Pour aller plus loin

- **Comparer les lots dans MLflow** : expérience `employee_attrition_monitoring`, tracer `psi_max` ou `auc` par run.
- **Changer de méthode** : `DataDriftPreset()` sans argument laisse Evidently choisir (tests KS / Chi² jusqu'à 1 000 lignes, distances de Wasserstein / Jensen-Shannon au-delà) ; `per_column_method` et `per_column_threshold` permettent un réglage par variable.
- **Pondérer par l'importance** : une dérive sur une variable peu utilisée par le modèle compte moins. Croiser le PSI avec `model[-1].feature_importances_`.
- **Rafraîchir la référence** au moment du réentraînement, pas avant.